In [ ]:
### 0 ### Inizializzazione - eseguire solo una volta per "vita" del kernel
import sys
import logging
from pathlib import Path

project_root = Path("~/tesi_graphrag").expanduser().resolve()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.pipeline_handler import PipelineHandler

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(name)s: %(message)s",
    datefmt="%H:%M:%S",
    force=True,
)

# Percorsi
sidecar_path = project_root / "data/processed/armstrong/full_test1.json"
queries_path = project_root / "data/queries/armstrong/eval_queries.json"

# Inizializzazione gestore pipeline
handler = PipelineHandler(
    sidecar_path=sidecar_path,
    queries_path=queries_path
)

print("!>> PipelineHandler e SidecarManager inizializzati con successo.")

In [ ]:
### PRE-TAGGING ###
candidate_tags = ["Jazz", "Ciclismo", "Astronautica"]
batch_size = 100

handler.pretag_sidecar(
        sidecar_path=sidecar_path,
        candidate_tags=candidate_tags,
        batch_size=batch_size
    )

In [ ]:
### CONFIG. PRE-DET. QUERIES ###
pred_text = "Chi è Armstrong e quali sono i principali traguardi raggiunti nella sua carriera?"
query_id = "Q_AMBIG_01"

handler.add_predetermined_query(
        query_text=pred_text,
        query_id=query_id
    )

In [ ]:
### 1 ### Impostazione Parametri
handler.set_parameters(
    top_j=5,
    top_k=7,
    top_n=5,
    order_by=["DIST", "TAGS"]
)

In [ ]:
### 2 ### Selezione Query

#Q_AMBIG_01, Q_AMBIG_02, Q_AMBIG_03, Q_SPEC_JAZZ_01, Q_SPEC_BIKE_01, Q_SPEC_ASTRO_01
handler.update_query(new_id="Q_AMBIG_01")

#handler.update_query(new_text="Quali sono i contributi musicali di Louis Armstrong e che strumento suonava?")

In [ ]:
### 3 ### Tag query
conv_tag_list = ["Astronautica"]
ass_tag_list = [""]
auto_ass_flag = True

handler.update_query_tags(
    new_conv_list=conv_tag_list,
    new_ass_list=ass_tag_list,
    auto_ass_flag=auto_ass_flag,
    print_debug=True
)

In [ ]:
### 4 ### Retrieval Vettoriale
handler.run_vectorial_retrieval()

In [ ]:
### 5 ### Reranking
handler.run_top_j_distance_tag_reranking(print_debug=True)

In [ ]:
import ipywidgets as widgets
from IPython.display import display

print("I Grafi che contano veramente nella pipeline finale (dunque escludendo quelli presenti per DEBUG) sono:")
print("  > 1° Retrieval (Vettoriale)")
print("  > Top-J Reranked")

# Mappatura tra i nomi visibili nei pulsanti e le chiavi interne delle fasi
PHASE_MAP = {
    "1° Retrieval (Vettoriale)": "vectorial",
    "2° Retrieval (Distanze)": "distance",
    "Top-J Reranked": "reranked",
    "All Reranked": "all_reranked"
}

# Selettore orizzontale delle fasi (ToggleButtons)
phase_selector = widgets.ToggleButtons(
    options=list(PHASE_MAP.keys()),
    description='Fase Grafo:',
    disabled=False,
    button_style='info',
    tooltips=[
        'Mostra i Top-K nodi del retrieval vettoriale iniziale',
        'Mostra i nodi iniziali uniti ai vicini recuperati per distanza',
        'Mostra i Top-J nodi selezionati dal Reranker per l\'LLM',
        'Mostra l\'intero elenco ordinato dal Reranker'
    ],
)

# Container per il rendering dinamico del widget
output_area = widgets.Output()

def on_phase_change(change):
    selected_label = change['new']
    phase_key = PHASE_MAP[selected_label]
    
    with output_area:
        output_area.clear_output(wait=True)
        handler.update_widget_view(phase_key)
        display(handler.widget)

# Registrazione dell'evento sul cambio di selezione
phase_selector.observe(on_phase_change, names='value')

# Inizializzazione della vista sulla prima fase
handler.update_widget_view("vectorial")

# Visualizzazione dell'interfaccia a schermo
display(phase_selector)
display(output_area)

# Popola l'output area con la vista iniziale del grafo
with output_area:
    display(handler.widget)

In [ ]:
### LLM ### Generazione risposte LLM
comparison = handler.compare_retrieval_vs_reranking()

baseline = comparison["vectorial_baseline"]
reranked = comparison["post_reranking"]

print("=" * 80)
print(f"???> Query: '{baseline['query']}'")
print("=" * 80)

print("\n --- 1. RISPOSTA BASELINE (1° Retrieval Vettoriale / Raw Records) ---")
print(f"  >> Chunk inclusi ({len(baseline['chunk_ids'])}): {baseline['chunk_ids']}")
print("\n  >> PROMPT INVIATO:")
print("-" * 40)
print(baseline["prompt"])
print("-" * 40)
print("\n  << RISPOSTA LLM:")
print(baseline["response"])

print("\n" + "=" * 80)

print("\n --- 2. RISPOSTA RERANKED (Post DistanceTagReranking / Top-J) ---")
print(f"  >> Chunk inclusi ({len(reranked['chunk_ids'])}): {reranked['chunk_ids']}")
print("\n  >> PROMPT INVIATO:")
print("-" * 40)
print(reranked["prompt"])
print("-" * 40)
print("\n  << RISPOSTA LLM:")
print(reranked["response"])
print("=" * 80)